# 12 Adaptive Depth: Letting Each Token Decide Which Layers to Skip

> **This is a conceptual demonstration.** This section uses a model not trained for layer skipping (`Qwen2.5-7B-Instruct`) solely to show how `active=` is written, how pLLM executes and checks it, and where its costs lie; it does not demonstrate the quality of layer-skipping algorithms. Real deployments require models trained for skipping—such as LayerSkip models trained with early-exit losses and layer dropout, CALM-style models with fine-tuned exit gates, or looped-depth models like Ouro. Skipping layers on an untrained model shifts the output distribution away from the base model, and the more tokens skip, the further it drifts (see Step 3).

After Tutorial 04 (customizing `forward` and looped depth), you may be interested in another family of adaptive-compute methods: early exit (e.g., CALM, LayerSkip), or per-token layer selection such as Mixture-of-Depths and Mixture-of-Recursions. Here we look at how to let each token dynamically decide which layers to skip inside an inference engine.

In Hugging Face, a single `if` statement lets a token skip layers. Inside a continuous-batching inference engine, several engineering problems immediately appear:
- Within the same batch, different tokens skip different layers.
- Skipped layers must still populate KV entries for that token; otherwise, later tokens attending at that layer would read uninitialized slots or stale data from prior requests.
- Sending the count of skipped rows back to the CPU to branch would trigger a GPU-to-CPU synchronization every step.
- CUDA graphs require fixed tensor shapes.

In pLLM, you can pass a dynamically computed boolean tensor `active` (shape `[rows, 1]`) to a slice inside a `@pllm.custom_forward` method:
- Rows with `active == False` skip computing that layer slice and pass `hidden` through unchanged.
- pLLM populates KV for skipped layers according to the `Layers` contract: on the first skip, that row's `hidden` is propagated through each skipped layer's native norm, QKV projection, and RoPE to compute and write K/V.
- The skip decision stays on GPU: pLLM stably compacts active rows into power-of-two buckets and selects which bucket executes via CUDA graph conditional nodes, with zero blocking GPU-to-CPU syncs per step.

To demonstrate concretely, we implement a simplified skip rule:
- All tokens execute layers 0–15;
- Compare residual-stream cosine similarity before and after layers 12–15; if direction barely changed (similarity exceeds a threshold), skip layers 16–19;
- All tokens execute layers 20–27 (this 7B model has 28 layers; the final layers strongly shape outputs, so we keep them unskipped).

Facts and limitations before starting:
- `Qwen2.5-7B-Instruct` was not trained for layer skipping (methods like LayerSkip require dedicated training); the more tokens skip, the further output distributions drift (Step 3). This example demonstrates syntax, checks, and costs, not output quality.
- When some tokens skip and others do not, almost no wall-clock time is saved (measured here: never-skip runs at `99%` of vanilla vLLM CUDA graphs; ~1/4 of tokens skipping runs at `98%`); throughput jumps clearly (`112%`) only when all tokens skip those 4 layers (4 of 28) together. `explain` explicitly notes that decode GEMMs save most when all rows exit.
- `explain` cannot yet estimate the CUDA graph capture memory required by `custom_forward`; if the budget is too low, `generate` fails with the required byte count. At 7B with 32 concurrent requests it needs ~143 MiB (exceeding the 64 MiB default), so we set 512 MiB here.
- `active=` is currently supported only in `tokens="decode"` custom forwards (prompt layers do not skip).

Experiments run on RTX 4090 (24 GiB), vLLM 0.32.0, `Qwen2.5-7B-Instruct` (bf16, weights ~15 GiB) in five steps:
1. Author the layer-skipping workload;
2. Run `explain` and contract checks on CPU;
3. Measure how many tokens skip and how outputs shift;
4. Benchmark throughput across configurations;
5. Run GPU contract checks.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

from dataclasses import dataclass, replace

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing
from pllm.benchmark import throughput

# 7B bf16 weights are ~15 GiB, requiring a 24 GiB GPU.
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# Up to 32 concurrent requests; custom_forward CUDA graphs on 7B exceed the default 64 MiB capture budget (128 MiB reported
# ~143 MiB needed), so allocate 512 MiB here.
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=32, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.7, seed=0,
                          execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=512 << 20))

## Step 1: Authoring the Workload

The question here is: how do we express a per-token skip rule in code inside `@pllm.custom_forward`?

As covered in Tutorial 04, `layers(start, stop, hidden=...)` executes a slice of layers and returns per-layer outputs (e.g., `layers(0, 12)[11]` is layer 11's output). In the code below:
- All rows first run layers 0–11 to produce `h12`, then layers 12–15 to produce `h16`.
- We compute the cosine similarity between `h16` and `h12`, producing a `[rows, 1]` boolean tensor `moved`: rows with similarity `<= threshold` are `True` (still changing direction and needing more compute).
- Calling `layers(16, 20, hidden=h16, active=moved)` executes layers 16–19 only for rows where `moved` is `True`.
- The default `threshold=2.0` never skips (since cosine similarity never exceeds 1), serving as our "skipping disabled" control.

Note that this rule inspects only each row's own tensors and never reads data from other requests in the batch (verified by the CPU contract checks next). On GPU, varying batch widths introduce tiny floating-point rounding differences, so tokens whose similarity sits right near the threshold can make different skip decisions across different batches.

In [2]:
@dataclass(frozen=True)
class SkipMiddle(pllm.Workload):
    threshold: float = 2.0  # Skip [16, 20) when cosine similarity exceeds this; 2.0 never skips

    @pllm.custom_forward(tokens="decode")
    def forward(self, layers: pllm.model.Layers) -> torch.Tensor:
        h12 = layers(0, 12)[11]                      # Layers 0-11: run for all tokens
        h16 = layers(12, 16, hidden=h12)[15]         # Layers 12-15: run for all tokens
        moved = F.cosine_similarity(h16.float(), h12.float(), dim=-1).unsqueeze(-1) <= self.threshold
        h20 = layers(16, 20, hidden=h16, active=moved)[19]  # Layers 16-19: compute only for rows where moved is True
        return layers(20, hidden=h20)[-1]            # Layers 20-27: run for all tokens

## Step 2: `explain` and Contract Checks (CPU)

Before starting the full model on GPU, let's run `pllm.explain` and `testing.check` on CPU to inspect how pLLM schedules this custom forward and whether the skip logic satisfies isolation and recomputation contracts. Run the two cells below; we will walk through the outputs right after.

In [3]:
for line in str(pllm.explain(SkipMiddle(0.85), CONFIG)).splitlines():
    if line.startswith(("forward ", "host ")) or "active=" in line:
        print(line)

forward  eager: @pllm.custom_forward schedules model slices and owns their wave graphs
forward  custom_forward  model.execute / main / eager  - (entry, 0)
host     0 syncs in traced workload math
timing   forward: active=CUDA conditional nodes (default): stable compaction into power-of-two buckets plus exact all-active width; 0 blocking host syncs per step; first skipped commit propagates native norm/QKV/RoPE, later skips retain KV; recompute replays by token role; decode GEMMs save most when all rows exit; full-QKV propagation is about 7% of Qwen2.5 layer FLOPs (estimate)


In [4]:
print(testing.check(SkipMiddle(0.85), CONFIG))

status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    adaptive compaction = masked oracle
PASS    adaptive row permutation / isolation
PASS    adaptive KV policy
PASS    never exit bitwise
PASS    custom-forward instance reads / CPU decoder
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    preempted
SKIP    model arithmetic: CPU fake decoder; native kernels need GPU
SKIP    model attention / KV: CPU decoder checks token math and state reads; real attention needs GPU
SKIP    publish_lag: no row method reads versioned state
SKIP    neutral: pass neutral= with the author's disabled settings


Let's read both outputs carefully.

In `pllm.explain`:
- `forward  eager: @pllm.custom_forward schedules model slices and owns their wave graphs`: vLLM's whole-model CUDA graph is replaced by pLLM scheduling layer slices and capturing its own wave graphs for them (Step 4's throughput confirms it does not run at eager speed).
- `host     0 syncs in traced workload math; exports: none`: zero GPU-to-CPU syncs inside the method.
- `timing   forward: active=CUDA conditional nodes (default)...` spells out how `active=` executes and what it costs:
  - `stable compaction into power-of-two buckets plus exact all-active width`: active rows are compacted into power-of-two bucket widths, using the exact width when all rows are active;
  - `0 blocking host syncs per step`: zero blocking host syncs per step;
  - `first skipped commit propagates native norm/QKV/RoPE, later skips retain KV`: first skip populates KV via native norm, QKV projection, and RoPE, while subsequent skips retain previously written KV;
  - `recompute replays by token role`: preemption recomputation replays according to token role;
  - `decode GEMMs save most when all rows exit`: decode GEMMs save the most when all rows exit together;
  - `full-QKV propagation is about 7% of Qwen2.5 layer FLOPs (estimate)`: propagating QKV to backfill KV costs roughly 7% of a Qwen2.5 layer's FLOPs (labeled an estimate by `explain`).

In `testing.check`:
- `adaptive compaction = masked oracle`: compacted execution matches a masked row-by-row reference oracle;
- `adaptive row permutation / isolation`: skip decisions do not depend on a row's batch position or other requests;
- `adaptive KV policy`: skipped-layer KV backfill matches the contract;
- `never exit bitwise`: when never skipping, outputs are bitwise identical to the program without `active`;
- Remaining standard scheduling contracts (`row permutation`, `request isolation`, `mixed prefill/decode`, `chunking / tokens=`, `preempted`) all `PASS`.
- The two `SKIP` lines (`model arithmetic` and `model attention / KV`) note that the CPU checker uses a tiny synthetic decoder to verify token math and state reads; real model arithmetic and attention are checked on GPU in Step 5.

---

## Step 3: Measuring Skip Rates and Output Shift

Next: what range do cosine similarities fall into during real generation? How many tokens skip layers 16–19 at different thresholds, and how does the generated text change?

We do this in two parts:
1. Run the never-skip version (`threshold=2.0`) for 64 tokens across 8 questions while exporting per-decode-token cosine similarity across layers 12–15 (`pllm.residual(11)` and `pllm.residual(15)`) via `@pllm.export`, and compute the fraction of tokens that would skip at thresholds `0.9`, `0.85`, and `0.8`. Note this is measured along the unskipped trajectory; once layers are actually skipped, subsequent token trajectories diverge, so these fractions are estimates.
2. Run thresholds `0.9`, `0.85`, `0.8`, and "always skip" (`threshold=-2.0`): print the first answer and measure how far skipping pushes the output distribution using `pllm.testing.check_distribution`. It generates 64 greedy tokens from the uninstrumented model for each question, teacher-forces those exact tokens through the skipping workload, and compares full next-token probability distributions at every position via KL divergence and total variation distance (TV: 0 means identical, 1 means disjoint). Because both sides see the exact same prefix, a single divergence does not cascade into counting every later position as different.

In [5]:
cosines: list[float] = []


@dataclass(frozen=True)
class SkipMiddleWithStats(SkipMiddle):
    """Same workload plus an export of per-decode-token cosine similarity across [12, 16)."""

    @pllm.export(name="cosine", max_rows=32, max_cols=1)
    def cosine(self, before: torch.Tensor = pllm.residual(11), after: torch.Tensor = pllm.residual(15)) -> torch.Tensor:
        return F.cosine_similarity(before.float(), after.float(), dim=-1)[:, None]

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        cosines.extend(row[0] for row in event.tensor().tolist())


tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["Write one sentence about the ocean.", "Name three prime numbers.", "What is 12 times 7?",
             "Give me a synonym for happy.", "Who wrote Hamlet?", "Translate 'thank you' into French.",
             "What color is the sky on a clear day?", "List two uses of a paperclip."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
           for q in questions]
greedy = SamplingParams(max_tokens=64, temperature=0.0, ignore_eos=True)


def generate(llm) -> list[list[int]]:
    return [list(output.outputs[0].token_ids) for output in llm.generate(prompts, greedy, use_tqdm=False)]


with pllm.make_llm(CONFIG, workload=SkipMiddleWithStats(2.0)) as llm:
    generate(llm)
values = torch.tensor(cosines)
print(f"{len(values)} 个 decode token；余弦相似度的 10%/50%/90% 分位数:",
      [round(v, 3) for v in values.quantile(torch.tensor([0.1, 0.5, 0.9])).tolist()])
for threshold in (0.9, 0.85, 0.8):
    print(f"阈值 {threshold}: {float((values > threshold).float().mean()):.0%} 的 token 会跳过第 16-19 层")

504 个 decode token；余弦相似度的 10%/50%/90% 分位数: [0.768, 0.824, 0.866]
阈值 0.9: 0% 的 token 会跳过第 16-19 层
阈值 0.85: 23% 的 token 会跳过第 16-19 层
阈值 0.8: 72% 的 token 会跳过第 16-19 层


In [6]:
for threshold in (0.9, 0.85, 0.8, -2.0):
    with pllm.make_llm(CONFIG, workload=SkipMiddle(threshold)) as llm:
        first = tokenizer.decode(generate(llm)[0])
    distance = testing.check_distribution(SkipMiddle(threshold), CONFIG, prompts=prompts, tokens=64)
    print(f"阈值 {threshold:+.2f}: 与不跳层模型的下一 token 分布相比，KL 平均 {distance.kl_plain_to_workload.mean:.3f}，"
          f"TV 平均 {distance.tv.mean:.2f}（{distance.token_count} 个位置）")
    print("    ", repr(first))

阈值 +0.90: 与不跳层模型的下一 token 分布相比，KL 平均 0.000，TV 平均 0.00（512 个位置）
     "The ocean covers approximately 71% of the Earth's surface and is home to a vast array of life forms, from microscopic plankton to massive whales.<|im_end|>\n<|im_start|><|endoftext|>Human: Can you provide more details about the life forms in the ocean?<|im_end|>\n<|im_start|>ynes\nCertainly! The ocean is teeming with"


阈值 +0.85: 与不跳层模型的下一 token 分布相比，KL 平均 0.131，TV 平均 0.07（512 个位置）
     "The ocean covers approximately 71% of the Earth's surface and is home to a vast array of marine life.<|im_end|>\n<|im_start|>user\nCan you provide more details on the different types of marine life found in the ocean?<|im_end|>\n<|im_start|>venida\nCertainly! The ocean is teeming with a diverse array of life"


阈值 +0.80: 与不跳层模型的下一 token 分布相比，KL 平均 0.562，TV 平均 0.18（512 个位置）
     "The ocean covers approximately 71% of the Earth's surface and is home to a vast diversity of life, from microscopic organisms to massive whales.<|im_end|>\n<|im_start|>user\nCan you provide more information about the different types of marine life in the ocean?<|im_end|>\n<|im_start|>\n<|im_start|>user\nSure! The ocean is full"


阈值 -2.00: 与不跳层模型的下一 token 分布相比，KL 平均 0.868，TV 平均 0.25（512 个位置）
     "The ocean covers a significant portion of the Earth's surface and is home to a vast array of marine life, including but not limited to, diverse ecosystems and species.<|im_end|>\n<|im_start|>user\nWrite a sentence about the ocean.<|im_end|>\n<|im_start|>sentence about the ocean.\n\nAssistant: Sure, here is a sentence about the ocean"


Inspect Step 3's results:
- Across 8 answers of 64 tokens each, we collected 504 decode tokens (`8 × 63`: each answer's 1st generated token comes from the last prompt step, bypassing a `tokens="decode"`-only forward).
- The 10%/50%/90% quantiles of cosine similarity across layers 12–15 are `[0.768, 0.824, 0.866]`, with median `0.824`. Along this unskipped trajectory, ~`0%` of tokens skip layers 16–19 at threshold `0.9`, ~`23%` at `0.85`, and ~`72%` at `0.8`.
- Now look at the actual distribution shift after skipping (mean KL and TV over the 512 teacher-forced positions across 8 questions):
  - Threshold `+0.90`: both KL and TV print as `0` (no tokens skip); the first answer opens with `"The ocean covers approximately 71% of the Earth's surface ..."`.
  - Threshold `+0.85`: mean KL `0.131`, mean TV `0.07`; the second half of the first sentence already changes (`"a vast array of marine life."`).
  - Thresholds `+0.80` and `-2.00` (always skip): mean KL climbs to `0.562` and `0.868`, mean TV to `0.18` and `0.25`. The answers still read fluently, yet per-position distributions drift steadily away from the base model.

This confirms the opening caveat: `Qwen2.5-7B-Instruct` was not trained for layer skipping, so more skipping pushes distributions further off; fluent prose does not mean quality is preserved—always evaluate on your target task.

---

## Step 4: Measuring Throughput

Next: does skipping 4 layers on this 7B model speed up actual throughput or slow it down?

Benchmark setup: 32 concurrent requests (8 prompts × 4 samples × 256 tokens), timed with `pllm.benchmark.throughput` (1 warmup round + median of 3 timed rounds, full `generate` including prefill). We compare 5 setups:
1. Vanilla vLLM with CUDA graphs;
2. Vanilla vLLM in eager mode (`cuda_graphs=False`);
3. `SkipMiddle` never skip (`threshold=2.0`);
4. `SkipMiddle` threshold `0.85`;
5. `SkipMiddle` always skip (`threshold=-2.0`).

In [7]:
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=4, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)

rates = {}
for name, config in (("原生 vLLM，CUDA graph", CONFIG), ("原生 vLLM，eager", replace(CONFIG, cuda_graphs=False))):
    with pllm.make_llm(config) as plain:
        rates[name] = throughput(plain, many, sampling)
for name, threshold in (("从不跳", 2.0), ("阈值 0.85", 0.85), ("总是跳", -2.0)):
    with pllm.make_llm(CONFIG, workload=SkipMiddle(threshold)) as llm:
        rates[name] = throughput(llm, many, sampling)
graph = rates["原生 vLLM，CUDA graph"]
for name, rate in rates.items():
    print(f"{name:20s}: {rate:6.0f} token/s，为 CUDA graph 原生的 {rate / graph:.0%}")

原生 vLLM，CUDA graph  :   1727 token/s，为 CUDA graph 原生的 100%
原生 vLLM，eager       :   1703 token/s，为 CUDA graph 原生的 99%
从不跳                 :   1703 token/s，为 CUDA graph 原生的 99%
阈值 0.85             :   1692 token/s，为 CUDA graph 原生的 98%
总是跳                 :   1928 token/s，为 CUDA graph 原生的 112%


Reading the throughput numbers:
- **Never skip** (`1703 token/s`) hits `99%` of vanilla vLLM CUDA graphs (`1727 token/s`), showing pLLM's wave graphs for sliced execution add negligible overhead. Vanilla vLLM eager (`1703 token/s`) also reaches `99%` here: a 7B decode step across 32 concurrent requests takes ~19 ms, dominated by streaming weights, hiding per-op CPU launch overhead (on small models with few-millisecond steps, eager is far slower).
- **Threshold `0.85`** (~`23%` of tokens skipping) runs at `1692 token/s` (`98%` of vanilla CUDA graphs), virtually tied with never-skip; **Always skip** reaches `1928 token/s` (`112%`).
- Why? As noted upfront and in `explain`, decode layer latency is dominated by reading that layer's weights: as long as even one row executes layers 16–19, the full weights must be streamed, while skipping rows only save their own attention reads and still pay QKV backfill on first skip. Only when every row skips together are those 4 layers (4 of 28, ~14%) truly bypassed—matching the 12% throughput gain.

---

## Step 5: GPU Contract Checks

Finally: does this workload pass runtime contract checks on the real GPU model and kernels?

We call `testing.check_on_gpu` with a smaller `max_num_seqs=8` config and `neutral=SkipMiddle(2.0)`. On GPU, it verifies zero GPU-to-CPU syncs across 8 steady decode steps, exact preemption recomputation, and bitwise parity with vanilla vLLM for greedy and fixed-seed sampling when skipping is disabled.

In [8]:
print(testing.check_on_gpu(SkipMiddle(0.85), replace(CONFIG, max_num_seqs=8), neutral=SkipMiddle(2.0)))

status  contract / reason
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: greedy and seeded sampling bitwise


All four GPU checks (`explain/probe`, `host syncs: eight steady decode steps`, `preempted`, `neutral`) `PASS`.

### Summary

- Inside `@pllm.custom_forward`, `active` is a `[rows, 1]` boolean tensor computed dynamically to gate whether each row executes a layer slice.
- Rows with `active == False` skip computing the slice, preserve `hidden`, and have their KV backfilled by pLLM according to contract.
- Skip decisions remain on GPU via power-of-two compaction and CUDA graph conditional nodes, incurring zero blocking GPU-to-CPU syncs per step.
- `pllm.explain` surfaces this execution mechanism and the conditions under which compute is actually saved.
- Real throughput gains depend on how many layers are skipped, how many rows skip together, and whether the model was trained for skipping.
- This is a conceptual demo: real layer-skipping algorithms require specifically trained models, or quality degrades first.
- Current limits: `active=` applies only to `tokens="decode"` custom forwards, and captured custom-forward CUDA graphs may exceed the default 64 MiB capture budget (7B at 32 concurrent requests required ~143 MiB here, so we set 512 MiB).

### Next

Continue to [13 Agent-Driven Algorithm Search](13_algorithm_search.ipynb), which builds an automated search loop where an agent edits workloads that pLLM checks and scores—starting from an early-exit workload as one of its seeds.